# Notebook 06 — Spectral Two-Stream (RQ3-new)

Frequency-domain branch fused with ViT. The diagnostic (NB05) gave
spectral-only AUC **0.884** (vs edge 0.517) — the gate is open.

**Variants**
- ViT-only (control) | Spectral-only (CNN-2D, MLP-1D) | Fusion (ViT+CNN-2D, ViT+MLP-1D)

**Protocol** (identical to NB04): train from in21k, **5 seeds**, paired t-test
Fusion-vs-ViT-only, ablation (seed 42), robustness on the NB02 degradation grid
(spectrum recomputed on the degraded image). All numbers computed at run time.

**Honest expectations.** ViT alone is already ~96% on clean; the spectral branch
is unlikely to beat it on clean. Its value, if any, is *complementary* (fusion) or
under *robustness*. Note spectral features are fragile to JPEG/blur (frequency
attacks), so fusion may help clean but hurt robustness — that itself is a finding.

In [1]:
# ============================================================
# CELL 1: IMPORTS + REPRODUCIBILITY
# ============================================================
import os, io, gc, json, random, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import ViTModel, ViTImageProcessor
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy import stats
from tqdm.auto import tqdm

SEED = 42
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: no CUDA -> use Colab T4 GPU then Run all again.')

Device: cuda


In [2]:
# ============================================================
# CELL 2: MOUNT DRIVE (for saving artifacts)
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/luanvan_thac_si/notebook06_artifacts'
except Exception as e:
    DRIVE = '/content/nb06_artifacts'
    print('No Colab drive; using', DRIVE)
os.makedirs(DRIVE, exist_ok=True)

Mounted at /content/drive


In [3]:
# ============================================================
# CELL 3: CONFIGURATION
# ============================================================
def free_mem():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

CONFIG = {
    'hf_dataset_id' : 'dragonintelligence/CIFAKE-image-dataset',
    'model_id'      : 'google/vit-base-patch16-224-in21k',
    'n_per_class'   : 4000,
    'train_ratio'   : 0.70, 'val_ratio': 0.15, 'test_ratio': 0.15,
    'seed'          : SEED,
    'multiseed_seeds': [42, 1, 2, 3, 4],

    # training [NOT STATED IN PAPER -- mirror NB04 + new-branch LR]
    'batch_size'    : 32, 'eval_batch': 64, 'num_epochs': 5,
    'lr_vit'        : 1e-5,    # backbone
    'lr_branch'     : 1e-4,    # spectral branch + head (trained from scratch)
    'weight_decay'  : 0.05,
    'feat_dim'      : 256,     # spectral branch output width

    # degradation grid (same as NB02/NB04)
    'degradations'  : {
        'JPEG'     : {'levels': [90, 70, 50, 30]},
        'Noise'    : {'levels': [5, 10, 20, 30]},
        'Blur'     : {'levels': [3, 5, 7]},
        'Downscale': {'levels': [0.75, 0.5, 0.25]},
    },
    'output_dir'    : '/content/nb06_out',
    'drive_backup'  : DRIVE,
}
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(CONFIG['multiseed_seeds'])

[42, 1, 2, 3, 4]


In [4]:
# ============================================================
# CELL 4: LOAD CIFAKE + BUILD SAME SUBSET/SPLIT AS NOTEBOOK 01
# ============================================================
raw_ds = load_dataset(CONFIG['hf_dataset_id'])
lf = raw_ds['train'].features['label']
FAKE_ID = lf.names.index('FAKE'); REAL_ID = lf.names.index('REAL')
id2label = {i: n for i, n in enumerate(lf.names)}; label2id = {n: i for i, n in id2label.items()}

pool = {FAKE_ID: [], REAL_ID: []}
for split in ['train', 'test']:
    for item in tqdm(raw_ds[split], desc=f'Indexing {split}', leave=False):
        pool[item['label']].append(item['image'])
rng = np.random.default_rng(SEED); n = CONFIG['n_per_class']
fi = rng.choice(len(pool[FAKE_ID]), size=n, replace=False)
ri = rng.choice(len(pool[REAL_ID]), size=n, replace=False)
all_imgs = [pool[FAKE_ID][i] for i in fi] + [pool[REAL_ID][i] for i in ri]
all_lbls = [FAKE_ID]*n + [REAL_ID]*n
del pool; gc.collect()

imgs_tv, imgs_test, y_tv, y_test = train_test_split(
    all_imgs, all_lbls, test_size=CONFIG['test_ratio'], stratify=all_lbls, random_state=SEED)
vf = CONFIG['val_ratio']/(CONFIG['train_ratio']+CONFIG['val_ratio'])
imgs_train, imgs_val, y_train, y_val = train_test_split(
    imgs_tv, y_tv, test_size=vf, stratify=y_tv, random_state=SEED)
y_train = np.array(y_train); y_val = np.array(y_val); y_test = np.array(y_test)
print(f"train {len(imgs_train)}  val {len(imgs_val)}  test {len(imgs_test)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/488 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/42.1M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/8.41M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Indexing train:   0%|          | 0/100000 [00:00<?, ?it/s]

Indexing test:   0%|          | 0/20000 [00:00<?, ?it/s]

train 5600  val 1200  test 1200


In [5]:
# ============================================================
# CELL 5: DEGRADATION FUNCTIONS (same as Notebook 02/04)
# ============================================================
def degrade_jpeg(p, q):
    b = io.BytesIO(); p.save(b, format='JPEG', quality=int(q)); b.seek(0)
    return Image.open(b).convert('RGB').copy()
def degrade_noise(p, s, rng=None):
    a = np.asarray(p).astype(np.float32); r = rng if rng is not None else np.random.RandomState(0)
    return Image.fromarray(np.clip(a + r.normal(0, s, a.shape), 0, 255).astype(np.uint8))
def degrade_blur(p, k):
    k = int(k); k += (k % 2 == 0); return Image.fromarray(cv2.GaussianBlur(np.asarray(p), (k, k), 0))
def degrade_downscale(p, sc):
    w, h = p.size
    return p.resize((max(1,int(w*sc)), max(1,int(h*sc))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
DEGRADERS = {'JPEG': degrade_jpeg, 'Noise': degrade_noise, 'Blur': degrade_blur, 'Downscale': degrade_downscale}

In [6]:
# ============================================================
# CELL 6: SPECTRAL FEATURES (native 32x32 FFT -> 2D log-mag + 1D radial)
# ============================================================
def to_gray32(p):
    return np.asarray(p.convert('L').resize((32, 32), Image.BILINEAR), np.float32) / 255.0
def log_mag(g):
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g)))).astype(np.float32)  # 32x32

_yy, _xx = np.indices((32, 32)); _R = np.sqrt((_yy-16)**2 + (_xx-16)**2).astype(int)
_NR = np.bincount(_R.ravel()); N_BINS = len(_NR)
def radial(m):
    return (np.bincount(_R.ravel(), m.ravel()) / np.maximum(_NR, 1)).astype(np.float32)

def spec2d_of(imgs):
    out = np.zeros((len(imgs), 1, 32, 32), np.float32)
    for i, im in enumerate(tqdm(imgs, desc='spec2d', leave=False)):
        m = log_mag(to_gray32(im)); out[i, 0] = (m - m.mean())/(m.std()+1e-6)  # per-image standardize
    return out
def radial_of(imgs):
    return np.stack([radial(log_mag(to_gray32(im))) for im in tqdm(imgs, desc='radial', leave=False)])

# precompute CLEAN spectral inputs
S2_train, S2_val, S2_test = spec2d_of(imgs_train), spec2d_of(imgs_val), spec2d_of(imgs_test)
R_train_raw = radial_of(imgs_train); R_val_raw = radial_of(imgs_val); R_test_raw = radial_of(imgs_test)
_rmu, _rsd = R_train_raw.mean(0), R_train_raw.std(0) + 1e-6     # standardize radial by train stats
R_train = (R_train_raw-_rmu)/_rsd; R_val = (R_val_raw-_rmu)/_rsd; R_test = (R_test_raw-_rmu)/_rsd
print('spec2d', S2_train.shape, '| radial', R_train.shape, '| N_BINS', N_BINS)

def spec_inputs(imgs, branch):
    """Compute spectral inputs on the fly (used for degraded images)."""
    if branch == 'cnn2d': return torch.tensor(spec2d_of(imgs))
    return torch.tensor((radial_of(imgs)-_rmu)/_rsd, dtype=torch.float32)

spec2d:   0%|          | 0/5600 [00:00<?, ?it/s]

spec2d:   0%|          | 0/1200 [00:00<?, ?it/s]

spec2d:   0%|          | 0/1200 [00:00<?, ?it/s]

radial:   0%|          | 0/5600 [00:00<?, ?it/s]

radial:   0%|          | 0/1200 [00:00<?, ?it/s]

radial:   0%|          | 0/1200 [00:00<?, ?it/s]

spec2d (5600, 1, 32, 32) | radial (5600, 23) | N_BINS 23


In [7]:
# ============================================================
# CELL 7: ViT TRANSFORM + DATASET (returns pixel_values, spec, label)
# ============================================================
proc = ViTImageProcessor.from_pretrained(CONFIG['model_id'])
clean_tf = Compose([Resize((224, 224)), ToTensor(),
                    Normalize(mean=proc.image_mean, std=proc.image_std)])

class SpecDS(Dataset):
    def __init__(self, imgs, labels, spec):
        self.imgs, self.labels = imgs, labels
        self.spec = torch.as_tensor(spec, dtype=torch.float32)
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        return clean_tf(self.imgs[i].convert('RGB')), self.spec[i], int(self.labels[i])

def loaders(branch, seed):
    s_tr = S2_train if branch == 'cnn2d' else R_train
    s_va = S2_val   if branch == 'cnn2d' else R_val
    g = torch.Generator(); g.manual_seed(seed)
    tr = DataLoader(SpecDS(imgs_train, y_train, s_tr), batch_size=CONFIG['batch_size'],
                    shuffle=True, generator=g, num_workers=0)
    va = DataLoader(SpecDS(imgs_val, y_val, s_va), batch_size=CONFIG['eval_batch'],
                    shuffle=False, num_workers=0)
    return tr, va

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [8]:
# ============================================================
# CELL 8: MODELS -- spectral branches + two-stream ViT
# ============================================================
class SpectralCNN2D(nn.Module):
    """Small CNN on the 32x32 log-magnitude spectrum."""
    def __init__(self, feat):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),   # 16
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),  # 8
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, feat), nn.ReLU())
    def forward(self, x): return self.net(x)

class SpectralMLP1D(nn.Module):
    """MLP on the radial spectral profile."""
    def __init__(self, n_in, feat):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, 128), nn.ReLU(), nn.Dropout(0.2),
                                 nn.Linear(128, feat), nn.ReLU())
    def forward(self, x): return self.net(x)

class TwoStreamViTSpectral(nn.Module):
    def __init__(self, vit_source, branch='cnn2d', use_vit=True, use_spectral=True, feat=256):
        super().__init__()
        self.use_vit, self.use_spectral = use_vit, use_spectral
        d = 0
        if use_vit:
            self.vit = ViTModel.from_pretrained(vit_source); d += 768
        if use_spectral:
            self.branch = SpectralCNN2D(feat) if branch == 'cnn2d' else SpectralMLP1D(N_BINS, feat)
            d += feat
        self.head = nn.Linear(d, 2)
    def forward(self, pixel_values=None, spec=None):
        f = []
        if self.use_vit:      f.append(self.vit(pixel_values=pixel_values).last_hidden_state[:, 0])
        if self.use_spectral: f.append(self.branch(spec))
        return self.head(torch.cat(f, 1))

def param_groups(model):
    vit, other = [], []
    for nm, p in model.named_parameters():
        (vit if nm.startswith('vit.') else other).append(p)
    g = []
    if vit:   g.append({'params': vit,   'lr': CONFIG['lr_vit']})
    if other: g.append({'params': other, 'lr': CONFIG['lr_branch']})
    return g

In [9]:
# ============================================================
# CELL 9: TRAIN / EVAL HELPERS
# ============================================================
@torch.no_grad()
def predict_model(model, imgs, branch):
    """Inference on arbitrary PIL list (clean or degraded). Recomputes spec."""
    model.eval()
    spec = spec_inputs(imgs, branch) if model.use_spectral else None
    preds, scores = [], []
    bs = CONFIG['eval_batch']
    for s in range(0, len(imgs), bs):
        chunk = imgs[s:s+bs]
        pv = torch.stack([clean_tf(im.convert('RGB')) for im in chunk]).to(DEVICE) if model.use_vit else None
        sp = spec[s:s+bs].to(DEVICE) if model.use_spectral else None
        logit = model(pixel_values=pv, spec=sp)
        p = torch.softmax(logit.float(), 1).cpu().numpy()
        preds.append(p.argmax(1)); scores.append(p[:, FAKE_ID])
    return np.concatenate(preds), np.concatenate(scores)

def evaluate(model, imgs, labels, branch):
    pred, sc = predict_model(model, imgs, branch)
    return {'acc': accuracy_score(labels, pred)*100,
            'f1':  f1_score(labels, pred, pos_label=FAKE_ID)*100,
            'auc': roc_auc_score((labels == FAKE_ID).astype(int), sc)*100}

def train_model(branch='cnn2d', use_vit=True, use_spectral=True, tag='m', seed=42):
    free_mem(); set_seed(seed)
    model = TwoStreamViTSpectral(CONFIG['model_id'], branch, use_vit, use_spectral,
                                 CONFIG['feat_dim']).to(DEVICE)
    tr, va = loaders(branch, seed)
    opt = torch.optim.AdamW(param_groups(model), weight_decay=CONFIG['weight_decay'])
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == 'cuda')
    crit = nn.CrossEntropyLoss()
    best_f1, best_state = -1, None
    for ep in range(CONFIG['num_epochs']):
        model.train()
        for pv, sp, lb in tqdm(tr, desc=f'{tag} s{seed} ep{ep+1}', leave=False):
            pv = pv.to(DEVICE) if use_vit else None
            sp = sp.to(DEVICE) if use_spectral else None
            lb = lb.to(DEVICE)
            opt.zero_grad()
            with torch.autocast(device_type='cuda', enabled=DEVICE == 'cuda'):
                loss = crit(model(pixel_values=pv, spec=sp), lb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        # val
        model.eval(); vp, vt = [], []
        with torch.no_grad():
            for pv, sp, lb in va:
                pv = pv.to(DEVICE) if use_vit else None
                sp = sp.to(DEVICE) if use_spectral else None
                vp.append(model(pixel_values=pv, spec=sp).argmax(1).cpu().numpy()); vt.append(lb.numpy())
        f1 = f1_score(np.concatenate(vt), np.concatenate(vp), pos_label=FAKE_ID)*100
        if f1 > best_f1:
            best_f1 = f1; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f'  {tag} seed{seed}: best val F1={best_f1:.2f}%')
    return model

In [10]:
# ============================================================
# CELL 10: MULTI-SEED TRAINING (ViT-only, Fusion-CNN, Fusion-MLP)
# ============================================================
VARIANTS = {
    'ViT-only'    : dict(branch='cnn2d', use_vit=True,  use_spectral=False),
    'Fusion-CNN'  : dict(branch='cnn2d', use_vit=True,  use_spectral=True),
    'Fusion-MLP'  : dict(branch='mlp1d', use_vit=True,  use_spectral=True),
}
multiseed = {v: [] for v in VARIANTS}
seed_models = {}
for seed in CONFIG['multiseed_seeds']:
    print(f'\n######### SEED {seed} #########')
    for vname, kw in VARIANTS.items():
        mdl = train_model(tag=vname, seed=seed, **kw)
        m = evaluate(mdl, imgs_test, y_test, kw['branch']); m['seed'] = seed
        multiseed[vname].append(m)
        print(f'  >> {vname} seed{seed}: acc={m["acc"]:.2f}  f1={m["f1"]:.2f}  auc={m["auc"]:.2f}')
        if seed == CONFIG['seed']: seed_models[vname] = mdl
        else: del mdl; free_mem()
print('\n[OK] multi-seed done.')


######### SEED 42 #########


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_683/4212453444.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == 'cuda')


...

Fusion-MLP s4 ep4:   0%|          | 0/175 [00:00<?, ?it/s]

Fusion-MLP s4 ep5:   0%|          | 0/175 [00:00<?, ?it/s]

  Fusion-MLP seed4: best val F1=97.09%


radial:   0%|          | 0/1200 [00:00<?, ?it/s]

  >> Fusion-MLP seed4: acc=96.17  f1=96.13  auc=99.48

[OK] multi-seed done.


In [11]:
# ============================================================
# CELL 11: CLEAN TABLE (mean ± std) + PAIRED t-TEST vs ViT-only
# ============================================================
def agg(lst, k):
    a = np.array([m[k] for m in lst]); return a.mean(), a.std(ddof=1)
rows = []
for v in ['ViT-only', 'Fusion-CNN', 'Fusion-MLP']:
    am, asd = agg(multiseed[v], 'acc'); fm, fsd = agg(multiseed[v], 'f1'); um, usd = agg(multiseed[v], 'auc')
    rows.append([v, f'{am:.2f} ± {asd:.2f}', f'{fm:.2f} ± {fsd:.2f}', f'{um:.2f} ± {usd:.2f}'])
df = pd.DataFrame(rows, columns=['Config', 'Accuracy (%)', 'F1 (%)', 'AUC-ROC (%)'])
print(df.to_string(index=False))
df.to_csv(os.path.join(CONFIG['output_dir'], 'table_clean_spectral.csv'), index=False)

base = [m['acc'] for m in multiseed['ViT-only']]
print('\n--- Paired t-test on accuracy vs ViT-only (n=%d) ---' % len(base))
for v in ['Fusion-CNN', 'Fusion-MLP']:
    x = [m['acc'] for m in multiseed[v]]; t, p = stats.ttest_rel(x, base)
    print(f'  {v}: Δ={np.mean(x)-np.mean(base):+.2f}pp  t={t:.3f}  p={p:.4f}  '
          f'-> {"significant" if p < 0.05 else "NOT significant"}')

    Config Accuracy (%)       F1 (%)  AUC-ROC (%)
  ViT-only 96.88 ± 0.29 96.86 ± 0.31 99.63 ± 0.07
Fusion-CNN 97.00 ± 0.34 96.98 ± 0.35 99.61 ± 0.03
Fusion-MLP 96.65 ± 0.38 96.65 ± 0.39 99.49 ± 0.03

--- Paired t-test on accuracy vs ViT-only (n=5) ---
  Fusion-CNN: Δ=+0.12pp  t=0.527  p=0.6261  -> NOT significant
  Fusion-MLP: Δ=-0.23pp  t=-1.033  p=0.3598  -> NOT significant


In [12]:
# ============================================================
# CELL 12: ABLATION (seed 42) -- spectral-only ceilings + fusion
# ============================================================
spec_cnn = train_model(branch='cnn2d', use_vit=False, use_spectral=True, tag='Spectral-CNN-only', seed=CONFIG['seed'])
spec_mlp = train_model(branch='mlp1d', use_vit=False, use_spectral=True, tag='Spectral-MLP-only', seed=CONFIG['seed'])
abl = [
    ('ViT-only',           seed_models['ViT-only'],   'cnn2d'),
    ('Spectral-only CNN-2D', spec_cnn,                'cnn2d'),
    ('Spectral-only MLP-1D', spec_mlp,                'mlp1d'),
    ('Fusion ViT+CNN-2D',  seed_models['Fusion-CNN'], 'cnn2d'),
    ('Fusion ViT+MLP-1D',  seed_models['Fusion-MLP'], 'mlp1d'),
]
rows = []
for name, mdl, br in abl:
    m = evaluate(mdl, imgs_test, y_test, br)
    rows.append([name, f"{m['acc']:.2f}", f"{m['f1']:.2f}", f"{m['auc']:.2f}"])
df_abl = pd.DataFrame(rows, columns=['Variant', 'Accuracy (%)', 'F1 (%)', 'AUC-ROC (%)'])
print(df_abl.to_string(index=False))
df_abl.to_csv(os.path.join(CONFIG['output_dir'], 'table_ablation_spectral.csv'), index=False)

/tmp/ipykernel_683/4212453444.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == 'cuda')


Spectral-CNN-only s42 ep1:   0%|          | 0/175 [00:00<?, ?it/s]

Spectral-CNN-only s42 ep2:   0%|          | 0/175 [00:00<?, ?it/s]

Spectral-CNN-only s42 ep3:   0%|          | 0/175 [00:00<?, ?it/s]

Spectral-CNN-only s42 ep4:   0%|          | 0/175 [00:00<?, ?it/s]

...

spec2d:   0%|          | 0/1200 [00:00<?, ?it/s]

radial:   0%|          | 0/1200 [00:00<?, ?it/s]

spec2d:   0%|          | 0/1200 [00:00<?, ?it/s]

radial:   0%|          | 0/1200 [00:00<?, ?it/s]

             Variant Accuracy (%) F1 (%) AUC-ROC (%)
            ViT-only        96.58  96.49       99.68
Spectral-only CNN-2D        78.00  76.80       87.42
Spectral-only MLP-1D        81.92  82.20       88.53
   Fusion ViT+CNN-2D        96.92  96.90       99.61
   Fusion ViT+MLP-1D        97.08  97.07       99.52


In [13]:
# ============================================================
# CELL 13: ROBUSTNESS (seed 42) -- does the spectral branch help or HURT?
# ============================================================
# Spectrum is recomputed on the DEGRADED image. Watch JPEG/Blur: frequency
# attacks may make Fusion WORSE than ViT-only (expected, R2).
test_models = [('ViT-only', seed_models['ViT-only'], 'cnn2d'),
               ('Fusion-CNN', seed_models['Fusion-CNN'], 'cnn2d'),
               ('Fusion-MLP', seed_models['Fusion-MLP'], 'mlp1d')]
rec = []
for name, mdl, br in test_models:
    p, _ = predict_model(mdl, imgs_test, br)
    rec.append({'model': name, 'degradation': 'Clean', 'level': 0, 'acc': accuracy_score(y_test, p)*100})
for dt, conf in CONFIG['degradations'].items():
    for lv in tqdm(conf['levels'], desc=dt):
        if dt == 'Noise':
            rngd = np.random.RandomState(0); deg = [degrade_noise(im, lv, rng=rngd) for im in imgs_test]
        else:
            deg = [DEGRADERS[dt](im, lv) for im in imgs_test]
        for name, mdl, br in test_models:
            p, _ = predict_model(mdl, deg, br)
            rec.append({'model': name, 'degradation': dt, 'level': lv, 'acc': accuracy_score(y_test, p)*100})
df_rob = pd.DataFrame(rec)
df_rob.to_csv(os.path.join(CONFIG['output_dir'], 'table_rob_spectral.csv'), index=False)
piv = df_rob.pivot_table(index=['degradation', 'level'], columns='model', values='acc')
print(piv.round(2).to_string())

spec2d:   0%|          | 0/1200 [00:00<?, ?it/s]

radial:   0%|          | 0/1200 [00:00<?, ?it/s]

JPEG:   0%|          | 0/4 [00:00<?, ?it/s]

spec2d:   0%|          | 0/1200 [00:00<?, ?it/s]

radial:   0%|          | 0/1200 [00:00<?, ?it/s]

...

spec2d:   0%|          | 0/1200 [00:00<?, ?it/s]

radial:   0%|          | 0/1200 [00:00<?, ?it/s]

spec2d:   0%|          | 0/1200 [00:00<?, ?it/s]

radial:   0%|          | 0/1200 [00:00<?, ?it/s]

model              Fusion-CNN  Fusion-MLP  ViT-only
degradation level                                  
Blur        3.00        53.92       53.25     56.83
            5.00        53.08       52.75     56.58
            7.00        54.42       53.67     56.25
Clean       0.00        96.92       97.08     96.58
Downscale   0.25        61.92       59.67     58.25
            0.50        53.92       53.00     56.92
            0.75        54.17       53.33     56.00
JPEG        30.00       92.83       92.08     85.33
            50.00       88.50       84.67     94.17
            70.00       96.75       95.75     97.17
            90.00       96.92       96.58     96.50
Noise       5.00        94.00       92.17     78.50
            10.00       91.75       90.58     71.00
            20.00       79.58       76.67     54.58
            30.00       62.08       56.33     52.33


In [14]:
# ============================================================
# CELL 14: SAVE ARTIFACTS
# ============================================================
import shutil
torch.save(seed_models['Fusion-CNN'].state_dict(), os.path.join(CONFIG['output_dir'], 'fusion_cnn_seed42.pt'))
for f in os.listdir(CONFIG['output_dir']):
    shutil.copy(os.path.join(CONFIG['output_dir'], f), os.path.join(CONFIG['drive_backup'], f))
print('[OK] NB06 complete. Artifacts ->', CONFIG['drive_backup'])
print('Summary: see table_clean_spectral.csv (5.x.1), table_ablation_spectral.csv (5.x ablation),'
      ' table_rob_spectral.csv (5.x robustness).')

[OK] NB06 complete. Artifacts -> /content/drive/MyDrive/luanvan_thac_si/notebook06_artifacts
Summary: see table_clean_spectral.csv (5.x.1), table_ablation_spectral.csv (5.x ablation), table_rob_spectral.csv (5.x robustness).
